# Implement GRPO (Group Relative Policy Optimization) — Solution

**Difficulty**: 🟣 Expert

**Companies**: DeepMind, Anthropic, OpenAI, AI Startups

---

### Problem Statement

Group Relative Policy Optimization (GRPO) is the algorithm used to train DeepSeek-R1 and represents a significant simplification over PPO for RLHF. The key innovation: **GRPO eliminates the need for a separate value/critic model** by using the group of sampled completions from the same prompt as its own baseline.

For each prompt, GRPO:
1. Samples G completions from the current policy
2. Scores each completion with a reward model
3. Computes advantages as: A_i = (r_i - mean(r)) / std(r) within the group
4. Updates the policy using a clipped objective (like PPO) with a KL penalty against a reference policy

The group of samples from the same prompt provides the baseline that PPO needs a value model for. This makes GRPO simpler to implement, more memory-efficient, and often more stable.

Your task is to implement GRPO from scratch using a simple sequence generation task.

---

### Requirements

1. **grpo_loss** — Clipped policy gradient with group-relative advantages and KL penalty.
2. **Group Sampling** — For each prompt, sample G completions and compute group-normalized advantages.
3. **KL Penalty** — Penalize divergence from a reference policy.
4. **No Value Model** — Unlike PPO, GRPO should not use a learned value function.

---

### Constraints

- ✅ Advantages must be computed per-group (normalized within each prompt's completions).
- ✅ Must include clipped objective (like PPO) to prevent large updates.
- ✅ Must include KL divergence penalty against reference policy.
- ✅ Reward should increase over training.
- ❌ Do **not** use a value/critic model.
- ❌ Do **not** use any existing RL libraries.

---

<details>
  <summary>💡 Hint</summary>

  - For each prompt, generate G completions. Score all G, then normalize: A_i = (r_i - mean(r_1..r_G)) / (std(r_1..r_G) + eps).
  - The clipped objective is the same as PPO: min(ratio * A, clip(ratio, 1-eps, 1+eps) * A).
  - KL penalty: approximate KL as mean(log_pi - log_ref) per token, then average over the sequence.
  - Think of the group as providing a "free" baseline — if your completion is better than the group average, the advantage is positive.
  - The total loss is: -policy_objective + beta * KL_penalty.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

In [ ]:
# Task: Generate sequences that sum close to a target value.
# Each "prompt" is a target sum. The policy generates tokens to hit that target.
VOCAB_SIZE = 10
SEQ_LEN = 8
GROUP_SIZE = 8  # G completions per prompt


class PolicyModel(nn.Module):
    """Policy network: takes a prompt embedding and generates token logits."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)  # +1 for BOS
        self.prompt_proj = nn.Linear(1, embed_dim)  # project scalar prompt
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, prompt=None, hidden=None):
        """Returns logits: (batch, seq_len, vocab_size)."""
        x = self.embedding(input_ids)
        if prompt is not None and hidden is None:
            # Inject prompt information into initial hidden state
            hidden = self.prompt_proj(prompt.unsqueeze(-1)).unsqueeze(0)  # (1, batch, hidden)
            # Pad/project to match GRU hidden size
            hidden = F.pad(hidden, (0, self.rnn.hidden_size - hidden.size(-1)))
        h, hidden = self.rnn(x, hidden)
        return self.head(h), hidden


class RewardModel(nn.Module):
    """Reward = -|sum(tokens) - target|. Higher is better."""
    def forward(self, sequences, targets):
        """
        Args:
            sequences: (batch, seq_len) token IDs
            targets: (batch,) target sums
        Returns:
            rewards: (batch,)
        """
        sums = sequences.float().sum(dim=-1)
        return -torch.abs(sums - targets)


torch.manual_seed(42)
policy = PolicyModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

print(f"Policy params: {sum(p.numel() for p in policy.parameters()):,}")
print(f"No value model needed! (Unlike PPO)")

In [ ]:
def generate_group_completions(policy: nn.Module, prompts: torch.Tensor,
                                group_size: int, seq_len: int,
                                vocab_size: int) -> tuple:
    """
    For each prompt, generate `group_size` completions.
    
    Args:
        policy: The policy model.
        prompts: (num_prompts,) tensor of prompt values (target sums).
        group_size: Number of completions per prompt.
        seq_len: Length of each completion.
        vocab_size: Size of vocabulary.
    
    Returns:
        sequences: (num_prompts * group_size, seq_len) generated tokens
        log_probs: (num_prompts * group_size, seq_len) log probs of each token
        prompt_targets: (num_prompts * group_size,) repeated targets
    """
    num_prompts = prompts.size(0)
    total_batch = num_prompts * group_size

    # Repeat each prompt group_size times
    expanded_prompts = prompts.repeat_interleave(group_size)  # (total_batch,)

    # Start with BOS token
    current = torch.full((total_batch, 1), vocab_size, dtype=torch.long)
    sequences = []
    log_probs = []
    hidden = None

    for t in range(seq_len):
        if t == 0:
            logits, hidden = policy(current, expanded_prompts)
        else:
            logits, hidden = policy(current[:, -1:], hidden=hidden)

        last_logits = logits[:, -1, :]  # (total_batch, vocab_size)
        dist = torch.distributions.Categorical(logits=last_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        sequences.append(action)
        log_probs.append(log_prob)
        current = torch.cat([current, action.unsqueeze(1)], dim=1)

    sequences = torch.stack(sequences, dim=1)  # (total_batch, seq_len)
    log_probs = torch.stack(log_probs, dim=1)  # (total_batch, seq_len)

    return sequences, log_probs, expanded_prompts


def compute_group_advantages(rewards: torch.Tensor, group_size: int) -> torch.Tensor:
    """
    Compute group-relative advantages.
    
    For each group of `group_size` completions from the same prompt,
    normalize: A_i = (r_i - mean(r)) / (std(r) + eps)
    """
    num_prompts = rewards.size(0) // group_size
    # Reshape to (num_prompts, group_size)
    grouped = rewards.view(num_prompts, group_size)

    # Compute mean and std within each group
    group_mean = grouped.mean(dim=1, keepdim=True)  # (num_prompts, 1)
    group_std = grouped.std(dim=1, keepdim=True)    # (num_prompts, 1)

    # Normalize
    advantages = (grouped - group_mean) / (group_std + 1e-8)

    # Reshape back
    return advantages.view(-1)  # (num_prompts * group_size,)


def grpo_loss(policy_logps: torch.Tensor, old_logps: torch.Tensor,
              advantages: torch.Tensor, clip_epsilon: float,
              beta: float, ref_logps: torch.Tensor) -> dict:
    """
    Compute GRPO loss: clipped policy gradient + KL penalty.
    
    Args:
        policy_logps: (batch, seq_len) current policy log probs per token.
        old_logps: (batch, seq_len) old policy log probs (from generation).
        advantages: (batch,) group-relative advantages.
        clip_epsilon: PPO-style clipping parameter.
        beta: KL penalty coefficient.
        ref_logps: (batch, seq_len) reference policy log probs per token.
    
    Returns:
        Dict with 'policy_loss', 'kl_div', 'total_loss'.
    """
    # Per-token ratio
    ratio = torch.exp(policy_logps - old_logps)  # (batch, seq_len)

    # Expand advantages to token level: (batch, 1)
    adv = advantages.unsqueeze(1)  # (batch, 1)

    # Clipped objective (per token, then mean)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * adv
    policy_loss = torch.min(surr1, surr2).mean()

    # KL penalty: approximate KL(pi || ref) = E[log_pi - log_ref]
    kl_div = (policy_logps - ref_logps).mean()

    # Total loss: maximize policy objective, penalize KL
    total_loss = -policy_loss + beta * kl_div

    return {
        'policy_loss': policy_loss,
        'kl_div': kl_div,
        'total_loss': total_loss
    }

In [ ]:
# === Validation ===
torch.manual_seed(42)

policy = PolicyModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)

num_prompts = 8
num_iterations = 40

reward_history = []
kl_history = []

print("=== GRPO Training ===")
print(f"Group size: {GROUP_SIZE} completions per prompt")
print(f"No value model used!\n")

for iteration in range(num_iterations):
    # Random target sums for this batch of prompts
    prompts = torch.randint(20, 50, (num_prompts,)).float()

    # 1. Generate group completions
    policy.eval()
    with torch.no_grad():
        sequences, old_log_probs, targets = generate_group_completions(
            policy, prompts, GROUP_SIZE, SEQ_LEN, VOCAB_SIZE)

        # 2. Score with reward model
        rewards = reward_model(sequences, targets)

        # 3. Compute group-relative advantages
        advantages = compute_group_advantages(rewards, GROUP_SIZE)

        # Get reference log probs
        bos = torch.full((sequences.size(0), 1), VOCAB_SIZE, dtype=torch.long)
        ref_input = torch.cat([bos, sequences[:, :-1]], dim=1)
        ref_logits, _ = ref_policy(ref_input, prompts.repeat_interleave(GROUP_SIZE))
        ref_dist = torch.distributions.Categorical(logits=ref_logits)
        ref_log_probs = ref_dist.log_prob(sequences)

    # 4. GRPO update
    policy.train()
    # Get current policy log probs
    curr_input = torch.cat([bos, sequences[:, :-1]], dim=1)
    curr_logits, _ = policy(curr_input, prompts.repeat_interleave(GROUP_SIZE))
    curr_dist = torch.distributions.Categorical(logits=curr_logits)
    curr_log_probs = curr_dist.log_prob(sequences)

    loss_dict = grpo_loss(
        policy_logps=curr_log_probs,
        old_logps=old_log_probs,
        advantages=advantages,
        clip_epsilon=0.2,
        beta=0.01,
        ref_logps=ref_log_probs
    )

    optimizer.zero_grad()
    loss_dict['total_loss'].backward()
    optimizer.step()

    avg_reward = rewards.mean().item()
    kl = loss_dict['kl_div'].item()
    reward_history.append(avg_reward)
    kl_history.append(kl)

    if iteration % 8 == 0:
        print(f"Iter {iteration:3d} | Reward: {avg_reward:.2f} | KL: {kl:.4f}")

# Test 1: Reward should increase
print("\n=== Reward Increase Test ===")
early_reward = sum(reward_history[:5]) / 5
late_reward = sum(reward_history[-5:]) / 5
print(f"Early avg reward: {early_reward:.2f}")
print(f"Late avg reward:  {late_reward:.2f}")
assert late_reward > early_reward, "Average reward should increase over training!"
print("PASSED\n")

# Test 2: KL should stay bounded
print("=== KL Bounded Test ===")
max_kl = max(abs(k) for k in kl_history)
print(f"Max |KL|: {max_kl:.4f}")
assert max_kl < 50.0, "KL divergence should stay bounded!"
print("PASSED\n")

# Test 3: Verify no value model was used
print("=== No Value Model Test ===")
print("GRPO uses group-relative advantages, no critic/value network needed.")
print(f"Only model trained: PolicyModel with {sum(p.numel() for p in policy.parameters()):,} params")
print("PASSED\n")

print("All tests passed!")